<a href="https://colab.research.google.com/github/jessicalewinter/deep-learning-specialization/blob/main/course-notebooks/deep-architectures/pytorch_workshop_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch from zero to transfer learning

**Goal:** build, train, inspect, and perturb an image classifier using PyTorch. Run cells top-to-bottom on **Google Colab** with a GPU if available (`Runtime -> Change runtime type -> T4 GPU`).

**Agenda**

| Time | Topic |
|---|---|
| 00:00–00:30 | Tensors, autograd, `nn.Module` |
| 00:30–01:00 | Dataset, DataLoader, augmentation |
| 01:00–01:40 | Pretrained ResNet18 and transfer learning |
| 01:40–01:50 | Break |
| 01:50–02:50 | Train/validation loops, metrics, checkpoints |
| 02:50–03:30 | Experiments and adversarial images |
| 03:30–04:00 | Feature hooks, Grad-CAM, recap |

**Runtime strategy:** we train on a stratified subset of CIFAR-10, resize to 128×128, and freeze the pretrained backbone. Full fine-tuning and longer training are optional. Downloading pretrained weights requires network access. The dataset has 32×32 source images, so resizing does not create additional detail.

**Legend:** `TODO` marks exercises. Some cells contain optional extensions; do not wait for every optional experiment during class.

## 0. Setup

We install `torchinfo`, a lightweight package that prints Keras-style model summaries (layers, shapes, parameter counts). For reproducibility, we fix random seeds; GPU algorithms and Colab environments can still cause small variations.

In [ ]:
%pip install -q torchinfo

In [ ]:
import os, random, time, copy
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torchinfo import summary

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

# Keep class runtime predictable.
IMAGE_SIZE = 128
BATCH_SIZE = 64 if DEVICE.type == 'cuda' else 16
TRAIN_PER_CLASS = 500    # 5,000 training examples total
VAL_PER_CLASS = 100      # 1,000 validation examples total
EPOCHS = 2
NUM_WORKERS = 2 if os.name != 'nt' else 0
DATA_ROOT = Path('/content/data' if Path('/content').exists() else './data')
CHECKPOINT_PATH = Path('/content/resnet18_cifar10.pt' if Path('/content').exists() else './resnet18_cifar10.pt')

In [ ]:
!nvidia-smi

## 1. Tensors and automatic differentiation (00:00–00:30)

A `torch.Tensor` is an n-dimensional array that can live on a GPU and participate in automatic differentiation. Typical CNN image batches have shape `[N, C, H, W]`. `requires_grad=True` asks autograd to track operations so `.backward()` can compute derivatives.

**TensorFlow bridge:** `tf.Tensor` ≈ `torch.Tensor`, `tf.GradientTape` ≈ autograd + `.backward()`, and `tf.keras.Model` ≈ `nn.Module`.

In [ ]:
x = torch.randn(4, 3, 8, 8, device=DEVICE)
print('Shape:', x.shape, 'dtype:', x.dtype, 'device:', x.device)
print('Mean:', x.mean().item())
print('Flattened:', x.flatten(start_dim=1).shape)

z = torch.tensor(3.0, requires_grad=True)
y = z**2 + 2*z + 1
y.backward()
print('dy/dz at z=3:', z.grad.item())  # 8.0

### Exercise — Autograd and gradient accumulation (5 min)

Compute the derivative of `f(x) = x**3 + 2*x` at `x=2`. Then call `.backward()` a second time on a newly computed `f(x)` without resetting gradients. What changes?

> **Student task:** Complete the `TODO` markers. Compare your result with the expected behavior described above.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
# TODO: compute f(x), call backward, print x.grad
# TODO: compute f(x) again, call backward, print x.grad again


### Define a network with `nn.Module`

`nn.Module` registers submodules and parameters, and `forward()` defines the computation. Call `model(images)`, not `model.forward(images)`, so PyTorch hooks and module machinery run correctly.

**Inspecting a model:** `torchinfo.summary` is the PyTorch counterpart of Keras `model.summary()`. Because PyTorch builds the computation dynamically, layer shapes are only known once data flows through the model, so we pass an `input_size` and `summary` runs a dummy batch for us. We choose which columns to display with `col_names` (e.g. `input_size`, `output_size`, `num_params`, `trainable`), and `row_settings=('var_names',)` shows the attribute name of each layer (`features`, `classifier`, ...), which is how you access it in code (`small_model.classifier`).

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.features(x)
        return self.classifier(torch.flatten(x, 1))

small_model = SmallCNN().to(DEVICE)
summary(small_model, input_size=(4, 3, 32, 32),
        col_names=('input_size', 'output_size', 'num_params'),
        row_settings=('var_names',))

### Exercise — Modify the CNN (5 min)

Add a third convolution (32 → 64 channels) before adaptive pooling and adjust the classifier. Use `summary` to confirm the output shape `[4, 10]` and see how many parameters the new layer adds.

> **Student task:** Complete the `TODO` markers. Compare your result with the expected behavior described above.

In [ ]:
# TODO: define class ThreeLayerCNN(nn.Module)
# TODO: use summary(...) to confirm the output shape and the new parameter count


## 2. Dataset, DataLoader and augmentation (00:30–01:00)

`Dataset[i]` returns a single `(image, label)` pair. `DataLoader` handles batches, shuffling, and worker processes. Augmentation should be applied **only to training data**. Validation uses deterministic transforms.

Pretrained ImageNet weights expect ImageNet channel normalization. We resize the low-resolution CIFAR images to 128×128 as a compute/quality compromise.

In [ ]:
WEIGHTS = ResNet18_Weights.DEFAULT
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Two dataset instances ensure that validation is NOT randomly augmented.
train_base = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=train_transform)
val_base = datasets.CIFAR10(DATA_ROOT, train=True, download=False, transform=val_transform)
class_names = train_base.classes

# Stratified, disjoint train/validation subsets drawn from CIFAR-10's official training split.
rng = np.random.default_rng(SEED)
labels_array = np.asarray(train_base.targets)
train_indices, val_indices = [], []
for class_id in range(10):
    class_indices = rng.permutation(np.flatnonzero(labels_array == class_id))
    train_indices.extend(class_indices[:TRAIN_PER_CLASS].tolist())
    val_indices.extend(class_indices[TRAIN_PER_CLASS:TRAIN_PER_CLASS+VAL_PER_CLASS].tolist())
assert set(train_indices).isdisjoint(val_indices)

train_ds = Subset(train_base, train_indices)
val_ds = Subset(val_base, val_indices)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')
print('Train:', len(train_ds), 'Validation:', len(val_ds), 'Classes:', class_names)

In [ ]:
def denormalize(images):
    mean = torch.tensor(MEAN, device=images.device).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=images.device).view(1, 3, 1, 1)
    return (images * std + mean).clamp(0, 1)

images, labels = next(iter(train_loader))
print('Batch shapes:', images.shape, labels.shape)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, img, label in zip(axes.flat, denormalize(images[:10]), labels[:10]):
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(class_names[label.item()]); ax.axis('off')
plt.tight_layout(); plt.show()

### Exercise — Augmentation gallery (5 min)

Visualize the **same source image** six times under stochastic training transforms. Explain which transformations preserve its class and why validation should be deterministic.

> **Student task:** Complete the `TODO` markers. Compare your result with the expected behavior described above.

In [ ]:
# TODO: select an image from train_base.data (PIL conversion is helpful)
# TODO: apply train_transform six times and plot a 2x3 gallery


## 3. Transfer learning with ResNet18 (01:00–01:40)

We load ImageNet pretrained weights, freeze the feature extractor, and replace its 1000-class classifier with a 10-class head. Only the new head will be updated initially.

**Important:** `requires_grad=False` controls gradient tracking for parameters; `model.eval()` changes Dropout and BatchNorm behavior. These are different mechanisms.

The summary below follows the image through ResNet18: watch the spatial size shrink from 128×128 to 4×4 while the channels grow to 512. Parameter counts in parentheses belong to **frozen** layers, and the `Trainable` column reads `True`, `False`, or `Partial` (for containers with mixed children). `summary` runs its dummy batch in eval mode and restores the previous mode afterwards, so it does not disturb BatchNorm statistics.

In [ ]:
model = resnet18(weights=WEIGHTS)
print('Original classifier:', model.fc)
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(DEVICE)

# Frozen backbone: BatchNorm running statistics should not drift.
# Call model.train() for training, then immediately put frozen BN layers into eval mode.
def set_training_mode(module):
    module.train()
    for submodule in module.modules():
        if isinstance(submodule, nn.modules.batchnorm._BatchNorm):
            if not any(p.requires_grad for p in submodule.parameters(recurse=False)):
                submodule.eval()

summary(model, input_size=(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE),
        col_names=('input_size', 'output_size', 'num_params', 'trainable'),
        depth=1, row_settings=('var_names',))

### Exercise — Unfreeze the last residual block (5 min)

Unfreeze `layer4` but keep the earlier blocks frozen. Use `summary` with the `num_params` and `trainable` columns to check which blocks are trainable and how many trainable parameters there are. Without `input_size`, `summary` skips the forward pass and lists only the parameters. Then **re-freeze `layer4`** before the main training loop so everyone starts from the same baseline.

> **Student task:** Complete the `TODO` markers. Compare your result with the expected behavior described above.

In [ ]:
# TODO: set requires_grad=True for model.layer4 parameters
# TODO: show summary(model, ...) with the 'num_params' and 'trainable' columns (no input_size needed)


In [ ]:
# TODO: set requires_grad=False for model.layer4 parameters
# TODO: confirm only the fc head is trainable again (hint: summary(model, verbose=0).trainable_params)


## 4. Train, validate, and save a model (01:50–02:50)

A PyTorch training step is explicit: zero old gradients → forward pass → loss → backward pass → optimizer step. `CrossEntropyLoss` expects **raw logits**, not probabilities. In evaluation, use `model.eval()` and `torch.inference_mode()`.

We track loss weighted by batch size (so a short final batch does not skew the epoch average) and accuracy.

### Exercise — Implement the training loop (5 min)

Fill in the missing training operations. Return `(average_loss, accuracy)`. **Do not apply softmax** before cross entropy.

> **Student task:** Complete the `TODO` markers. Compare your result with the expected behavior described above.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    set_training_mode(model)
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        # TODO: clear gradients, forward, loss, backward, optimizer step
        # TODO: update total_loss, correct, total
    return total_loss / total, correct / total


### Exercise — Implement validation (5 min)

Evaluate without gradient tracking. Return the same metrics as the training function. What would happen if you forgot `model.eval()`?

> **Student task:** Complete the `TODO` markers. Compare your result with the expected behavior described above.

In [ ]:
def evaluate(model, loader, criterion, device):
    # TODO: evaluation mode and inference_mode context
    # TODO: compute weighted loss and accuracy
    pass


### Execute the training run

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=1e-3)
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = -1.0
best_state = None

for epoch in range(EPOCHS):
    start = time.perf_counter()
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
    for key, value in [('train_loss', train_loss), ('train_acc', train_acc),
                       ('val_loss', val_loss), ('val_acc', val_acc)]:
        history[key].append(value)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
    print(f'Epoch {epoch+1}/{EPOCHS} | train loss {train_loss:.3f} acc {train_acc:.1%} '
          f'| val loss {val_loss:.3f} acc {val_acc:.1%} '
          f'| {time.perf_counter()-start:.1f}s')

model.load_state_dict(best_state)
torch.save({'model_state_dict': best_state, 'class_names': class_names,
            'image_size': IMAGE_SIZE, 'history': history}, CHECKPOINT_PATH)
print('Saved best checkpoint:', CHECKPOINT_PATH)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history['train_loss'], marker='o', label='train')
axes[0].plot(history['val_loss'], marker='o', label='validation')
axes[0].set_title('Cross-entropy loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history['train_acc'], marker='o', label='train')
axes[1].plot(history['val_acc'], marker='o', label='validation')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout(); plt.show()

### Checkpoints and inference

PyTorch typically saves a `state_dict` (parameter and buffer tensors) instead of serializing a whole model object. To reload, recreate the architecture and load its state. `torch.load(..., weights_only=True)` is preferable when the checkpoint contains tensors and basic data types.

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
restored = resnet18(weights=None)
restored.fc = nn.Linear(restored.fc.in_features, len(checkpoint['class_names']))
restored.load_state_dict(checkpoint['model_state_dict'])
restored = restored.to(DEVICE).eval()
with torch.inference_mode():
    sample_logits = restored(images[:4].to(DEVICE))
    sample_probs = sample_logits.softmax(dim=1)
print('Predictions:', [class_names[i] for i in sample_probs.argmax(1).tolist()])
print('Confidence:', sample_probs.max(1).values.cpu().tolist())
# Confidence is not necessarily calibrated probability.

## 5. Experiments (02:50–03:30)

In pairs, choose one experiment. **Keep the validation split unchanged**. Record the change you made, runtime, and validation accuracy. Do not compare results from different splits.

Suggested challenges:

1. Unfreeze `layer4`, train with a smaller learning rate (e.g. `1e-4`), and compare with frozen feature extraction. Check the `Trainable` column of `summary` before creating the optimizer.
2. Modify `RandomRotation(10)` to `RandomRotation(45)`: does more augmentation always help?
3. Compare learning rates `1e-2`, `1e-3`, `1e-4` with a fresh classifier for each run.
4. Explore the input-gradient attack below.

For fair comparisons, recreate the model and optimizer for each trial, fix seeds, and keep the same subset and epoch budget. Running every experiment sequentially is not expected in class.

### FGSM: change a prediction using input gradients

Unlike ordinary training, we differentiate the loss **with respect to the input image**. We apply an untargeted Fast Gradient Sign Method perturbation in **pixel space**, bounded by `epsilon` on the `[0,1]` scale. The model receives normalized images, so we convert the pixel perturbation into normalized coordinates. This is an educational robustness demonstration, not a guarantee of a successful attack.

### Exercise — Implement FGSM (5 min)

Complete the FGSM function. It should not update the model parameters, and should return a valid normalized image corresponding to pixels in `[0,1]`.

> **Student task:** Complete the `TODO` markers. Compare your result with the expected behavior described above.

In [ ]:
def fgsm_attack(model, normalized_images, labels, epsilon=4/255):
    model.eval()
    x = normalized_images.detach().clone().requires_grad_(True)
    # TODO: forward + cross-entropy + input gradient
    # TODO: convert gradient to pixel-space direction
    # TODO: perturb pixels by epsilon and clamp to [0,1]
    # TODO: normalize again and return a detached tensor
    pass


In [ ]:
batch_images, batch_labels = next(iter(val_loader))
batch_images = batch_images[:8].to(DEVICE)
batch_labels = batch_labels[:8].to(DEVICE)
model.eval()
with torch.inference_mode():
    original_preds = model(batch_images).argmax(1)
adversarial_images = fgsm_attack(model, batch_images, batch_labels, epsilon=8/255)
with torch.inference_mode():
    attacked_preds = model(adversarial_images).argmax(1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
original_pixels = denormalize(batch_images).cpu()
attacked_pixels = denormalize(adversarial_images).cpu()
for i, ax in enumerate(axes.flat):
    ax.imshow(original_pixels[i].permute(1, 2, 0))
    ax.set_title(f'{class_names[original_preds[i]]} → {class_names[attacked_preds[i]]}', fontsize=9)
    ax.axis('off')
plt.suptitle('Original images; titles show prediction before → after FGSM')
plt.tight_layout(); plt.show()
print('Changed predictions:', (original_preds != attacked_preds).sum().item(), '/ 8')
print('Max pixel change:', (original_pixels-attacked_pixels).abs().max().item())

## 6. Model inspection, hooks and Grad-CAM (03:30–04:00)

Forward hooks let you capture intermediate activations without editing the model. The captured `layer4` activation should match the `layer4` output shape from the Section 3 summary (here with a batch of one image). Grad-CAM uses the gradient of a selected class logit with respect to feature maps in a convolutional layer, averages those gradients spatially to obtain channel weights, then forms a weighted activation map.

Grad-CAM is a **heuristic visualization**, not proof of causal reasoning. A highlighted region does not necessarily mean the model depends exclusively on that region.

In [ ]:
activations = {}
def capture_activation(module, inputs, output):
    activations['layer4'] = output.detach()

hook_handle = model.layer4.register_forward_hook(capture_activation)
model.eval()
with torch.inference_mode():
    _ = model(batch_images[:1])
print('layer4 activation shape:', activations['layer4'].shape)
hook_handle.remove()

### Exercise — Compute a Grad-CAM heatmap (5 min)

Implement a Grad-CAM heatmap for the predicted class of one input. Use a forward hook on `model.layer4`, and `torch.autograd.grad` to get gradients with respect to that activation. Resize the heatmap to input resolution.

> **Student task:** Complete the `TODO` markers. Compare your result with the expected behavior described above.

In [ ]:
import torch.nn.functional as F

def grad_cam(model, image):
    model.eval()
    # TODO: hook model.layer4 and keep its output (without detaching)
    # TODO: forward image and choose predicted class
    #       (hint: the backbone is frozen; what must require grad for layer4's output to be in the graph?)
    # TODO: get gradient of selected logit w.r.t. activations
    # TODO: channel-weighted sum, ReLU, bilinear upsample, normalize
    # TODO: remove hook and return 2D heatmap and class index
    pass


In [ ]:
num_examples = min(3, len(batch_images))
indices = torch.randperm(len(batch_images))[:num_examples]
fig, axes = plt.subplots(num_examples, 3, figsize=(12, 4 * num_examples), squeeze=False)
for row, idx in enumerate(indices.tolist()):
    input_img = batch_images[idx:idx + 1]
    heatmap, predicted_class = grad_cam(model, input_img)
    original = denormalize(input_img)[0].cpu().permute(1, 2, 0).numpy()
    axes[row, 0].imshow(original); axes[row, 0].set_title(f'Original (#{idx})')
    axes[row, 1].imshow(heatmap, cmap='jet', vmin=0, vmax=1); axes[row, 1].set_title('Grad-CAM')
    axes[row, 2].imshow(original)
    axes[row, 2].imshow(heatmap, cmap='jet', alpha=0.4, vmin=0, vmax=1)
    axes[row, 2].set_title('Overlay: ' + class_names[predicted_class])
for ax in axes.flat: ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Wrap-up and next steps

**Exit questions:**

- Why do gradients accumulate unless we clear them?
- How is freezing parameters different from switching to evaluation mode?
- Why should validation transforms be deterministic?
- What can input gradients and Grad-CAM tell us — and what can't they tell us?

**References:** [PyTorch tutorials](https://docs.pytorch.org/tutorials/), [torchvision ResNet18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html), [torchvision transforms](https://docs.pytorch.org/vision/stable/transforms.html).